# Citiz — 3. Indicateurs et emplacements candidats (Gold)
Runtime Python. Ne nécessite pas de lakehouse par défaut; accès par nom dans le même workspace.
Deux granularités: commune et emplacement. Rayon de chalandise ≠ isochrone piétonne.
Le score strict 0–100 n'est publié que si ses huit composantes sont connues; deux bornes
montrent l'incertitude. Un score provisoire sur les seuls critères connus (poids renormalisés)
est publié à côté, avec sa complétude: une présélection, pas un score final.
Les centres de carreaux servent seulement à une présélection. Les bassins se recouvrent:
ne jamais sommer leur population, ni leurs concurrents pour faire un total national.


In [ ]:
%pip install -q "deltalake>=1,<2" shapely pyproj scipy

In [ ]:
SILVER = 'silver'
GOLD = 'gold'
SCHEMA = 'dbo'
LOCAL_ROOT = None
GROWTH_START = 2017
GROWTH_END = 2023
CATCHMENT_M = 500
COMPETITION_RADIUS_M = 1000
CITIZ_SPACING_M = 500
WALK_MAX_MIN = 5
WALK_SPEED_KMH = 4.5
WALK_DETOUR_FACTOR = 1.3  # estimation explicite; ne remplace jamais un itinéraire piéton
MAX_AUTO_COMMUNES = 50
MAX_AUTO_CANDIDATES_PER_COMMUNE = 100
# Les candidats manuels sont tous conservés, sans limite.
MAX_INVENTORY_AGE_DAYS = 180
MAX_MEASUREMENT_AGE_DAYS = 730
MIN_POP_CATCHMENT = 500
STABLE_GROWTH_PCT_AN = -0.2
CONGESTION_MAX = 0.50  # +50 % de durée vs circulation fluide, seuil métier à valider
COMPETITORS_SATURATION = 8  # stations automobiles observées dans 1 km, proxy à valider
THEFT_ALERT_PER_1000 = 10  # vols de véhicules / 1000 habitants; pas une probabilité de sinistre
WEIGHTS = {'population':.25,'croissance':.10,'motorisation':.10,'transport':.15,
           'securite':.10,'concurrence':.10,'congestion':.10,'mobilite':.10}
# Aucun seuil ci-dessus ne constitue une calibration de rentabilité Citiz.


In [ ]:
import os, re, json, unicodedata, uuid, datetime as dt
import numpy as np
import pandas as pd
import pyarrow as pa
from deltalake import DeltaTable, write_deltalake
from scipy.spatial import cKDTree
RUN_ID = str(uuid.uuid4())
STATUS = []

def nu():
    import notebookutils
    return notebookutils

def root(lakehouse):
    if LOCAL_ROOT:
        return f"{LOCAL_ROOT}/{lakehouse}", None
    info = nu().lakehouse.get(lakehouse)
    if not isinstance(info, dict):
        info = json.loads(json.dumps(info, default=lambda x: x.__dict__))
    props = info.get("properties", {})
    path = props.get("abfsPath")
    if not path:
        raise RuntimeError(f"abfsPath absent pour {lakehouse}; contrôler notebookutils.lakehouse.get")
    return path.rstrip("/"), {"bearer_token": nu().credentials.getToken("storage"), "use_fabric_endpoint":"true"}

def table_path(lakehouse, name):
    p,o = root(lakehouse)
    return f"{p}/Tables/{SCHEMA + '/' if SCHEMA else ''}{name}",o

def read_table(lakehouse, name, required=False, run_id=None):
    p,o = table_path(lakehouse,name)
    try:
        t=DeltaTable(p,storage_options=o).to_pandas()
    except Exception as e:
        if required: raise RuntimeError(f"Table obligatoire illisible: {lakehouse}.{name}") from e
        print(f"Absente/illisible: {lakehouse}.{name}: {str(e)[:120]}")
        return pd.DataFrame()
    if run_id is not None:
        if '_run_id' not in t: return pd.DataFrame()
        t=t[t['_run_id']==run_id].copy()
    return t.drop(columns=['_run_id'],errors='ignore')

def write_table(lakehouse,name,df):
    df=df.reset_index(drop=True).copy()
    df['_run_id']=RUN_ID
    # Types explicites même pour colonnes entièrement nulles / tables vides.
    a=pa.Table.from_pandas(df,preserve_index=False)
    a=pa.Table.from_arrays([a[c].cast(pa.string()) if pa.types.is_null(a[c].type) else a[c]
                            for c in a.column_names],names=a.column_names)
    p,o=table_path(lakehouse,name)
    if LOCAL_ROOT: os.makedirs(os.path.dirname(p),exist_ok=True)
    write_deltalake(p,a,mode='overwrite',schema_mode='overwrite',storage_options=o)
    STATUS.append({'table':name,'lignes':len(df),'run_id':RUN_ID})
    print(f"{lakehouse}.{name}: {len(df):,} lignes")

def nnum(s):
    return pd.to_numeric(s.astype('string').str.replace('\u202f','',regex=False)
          .str.replace(' ','',regex=False).str.replace(',','.',regex=False),errors='coerce')

def textnorm(s):
    return s.fillna('').astype(str).map(lambda v: unicodedata.normalize('NFKD',v)
                    .encode('ascii','ignore').decode().lower().strip())

def codes(s):
    s=s.astype('string').str.strip().str.upper().str.replace(r'\.0$','',regex=True)
    return s.str.zfill(5).where(s.str.fullmatch(r'\d{1,5}|2[AB]\d{3}',na=False))

def arm_to_com(s):
    s=codes(s)
    return s.mask(s.str.match(r'751\d{2}',na=False),'75056').mask(s.str.match(r'132\d{2}',na=False),'13055').mask(s.str.match(r'6938[1-9]',na=False),'69123')

def assert_unique(df,keys,name):
    if df[keys].isna().any().any() or df.duplicated(keys).any():
        raise ValueError(f"Clé nulle ou dupliquée: {name} {keys}")

def valid_points(df):
    d=df.copy()
    d['lat']=nnum(d['lat']); d['lon']=nnum(d['lon'])
    return d[d.lat.between(-90,90)&d.lon.between(-180,180)&~((d.lat==0)&(d.lon==0))].copy()

def xyz(lon,lat):
    a,b=np.radians(np.asarray(lon,dtype=float)),np.radians(np.asarray(lat,dtype=float))
    return np.column_stack([np.cos(b)*np.cos(a),np.cos(b)*np.sin(a),np.sin(b)])

def nearest(q,refs):
    if refs.empty: return np.full(len(q),-1,dtype=int),np.full(len(q),np.nan)
    d,i=cKDTree(xyz(refs.lon,refs.lat)).query(xyz(q.lon,q.lat),k=1)
    return i,2*6371008.8*np.arcsin(np.clip(d/2,0,1))

def radius_sum(q,refs,radius,value=None):
    if refs.empty: return np.zeros(len(q))
    tree=cKDTree(xyz(refs.lon,refs.lat)); chord=2*np.sin(radius/(2*6371008.8))
    result=[]
    for start in range(0,len(q),1000):
        hits=tree.query_ball_point(xyz(q.lon.iloc[start:start+1000],q.lat.iloc[start:start+1000]),chord)
        result.extend([float(refs.iloc[h][value].sum()) if value else len(h) for h in hits])
    return np.array(result)


In [ ]:
run=read_table(SILVER,'pipeline_run',True)
if len(run)!=1 or run.iloc[0]['status']!='complete': raise ValueError('Silver non validé')
RUN_ID=run.iloc[0]['run_id']
def S(name,required=False): return read_table(SILVER,name,required,RUN_ID)
com=S('dim_commune',True); hist=S('fact_population',True)
grid=S('fact_carreau_population'); stops=S('dim_arret'); stations=S('dim_station')
cars=S('fact_motorisation'); crime=S('fact_delinquance'); flows=S('fact_flux')
k=com[['commune_code']].copy()
rows=[]
for cc,h in hist.groupby('commune_code'):
    ys=sorted(h.annee.unique()); y1=min(ys,key=lambda y:abs(y-GROWTH_END)); y0=min(ys,key=lambda y:abs(y-GROWTH_START))
    v=h.set_index('annee').population
    row={'commune_code':cc,'annee_population':int(y1),'population':v[y1],
         'annee_pop_debut':int(y0),'population_debut':v[y0], 'croissance_annuelle':np.nan}
    if y1-y0>=5 and v[y0]>0:
        row['croissance_annuelle']=100*((v[y1]/v[y0])**(1/(y1-y0))-1)
    rows.append(row)
k=k.merge(pd.DataFrame(rows),on='commune_code',how='left')
if len(cars): k=k.merge(cars.drop(columns='annee').rename(columns={'part_sans_voiture':'part_menages_sans_voiture'}),on='commune_code',how='left')
else: k['part_menages_sans_voiture']=np.nan
if len(flows):
    f=flows.copy(); f['nb_sortants']=f.nb_actifs.where(f.origine!=f.destination,0)
    a=f.groupby('origine')[['nb_actifs','nb_sortants']].sum().reset_index().rename(columns={'origine':'commune_code'})
    a['part_actifs_sortants']=a.nb_sortants/a.nb_actifs.replace(0,np.nan)
    k=k.merge(a,on='commune_code',how='left')
# Indicateurs SSMSI séparés, pour l'année complète la plus récente de la commune.
crime_rows=[]
if len(crime):
    for cc,h in crime.groupby('commune_code'):
        for year,g in h.groupby('annee',sort=True):
            if year<dt.date.today().year-3 or year>dt.date.today().year: continue
            row={'commune_code':cc,'annee_delinquance':year}
            for cat in ['vol_vehicule','vol_dans_vehicule','vol_accessoire','degradations_generales']:
                sub=g[g.categorie==cat]
                row[cat+'_pour_mille']=sub.iloc[0].taux_pour_mille if len(sub) else np.nan
            crime_rows.append(row)
    if crime_rows:
        a=pd.DataFrame(crime_rows).sort_values('annee_delinquance').drop_duplicates('commune_code',keep='last')
        k=k.merge(a,on='commune_code',how='left')
for c in ['vol_vehicule','vol_dans_vehicule','vol_accessoire','degradations_generales']:
    if c+'_pour_mille' not in k: k[c+'_pour_mille']=np.nan
mob=S('input_mobilite_mesuree')
if len(mob):
    mob=mob[mob.annee.between(dt.date.today().year-3,dt.date.today().year)].sort_values('annee').drop_duplicates('commune_code',keep='last')
    k=k.merge(mob.rename(columns={'annee':'annee_mobilite','source':'source_mobilite'}),on='commune_code',how='left')
else: k['temps_moyen_min']=np.nan
traf=S('input_congestion')
if len(traf):
    age=(pd.Timestamp.today().normalize()-pd.to_datetime(traf.date_fin)).dt.days
    traf=traf[age.between(0,MAX_MEASUREMENT_AGE_DAYS)]
    k=k.merge(traf.rename(columns={'source':'source_congestion'}),on='commune_code',how='left')
else: k['indice_congestion']=np.nan
for colname in ['menages','menages_sans_voiture','nb_actifs','nb_sortants','nb_navetteurs',
 'temps_moyen_min','indice_congestion','temps_pointe_min','temps_fluide_min','nb_observations']:
    if colname not in k: k[colname]=np.nan
write_table(GOLD,'dim_commune',com.drop(columns='contour_json'))
write_table(GOLD,'fact_population',hist)
write_table(GOLD,'fact_delinquance',crime if len(crime) else pd.DataFrame(columns=['commune_code','annee','categorie','nombre','taux_pour_mille']))
write_table(GOLD,'gold_kpi_commune',k)


In [ ]:
c=S('input_candidats')
if c.empty:
    # Présélection explicite, aucune optimisation d'implantation sur ces points théoriques.
    if grid.empty:
        c=pd.DataFrame(columns=['candidat_id','commune_code','nom','lat','lon','origine_candidat'])
    else:
        eligible=k[k.population.notna()].sort_values(['population','commune_code'],ascending=[False,True]).head(MAX_AUTO_COMMUNES)
        g=grid[grid.commune_code.isin(eligible.commune_code)].sort_values(['population','carreau_id'],ascending=[False,True])
        c=g.groupby('commune_code',sort=False).head(MAX_AUTO_CANDIDATES_PER_COMMUNE)[['carreau_id','commune_code','lat','lon']].rename(columns={'carreau_id':'candidat_id'})
        c['candidat_id']='grille:'+c.candidat_id.astype(str); c['nom']=c.candidat_id
        c['origine_candidat']='centre_carreau_theorique'
else: c['origine_candidat']='manuel'
c=c.reset_index(drop=True)
assert_unique(c,['candidat_id'],'candidats')
x=c[['candidat_id','commune_code']].copy()
x['population_bassin']=radius_sum(c,grid,CATCHMENT_M,'population') if len(grid) else np.nan
x['annee_population_bassin']=2021 if len(grid) else np.nan
x['nb_arrets_proches']=radius_sum(c,stops,CATCHMENT_M) if len(stops) else np.nan
idx,dist=nearest(c,stops) if len(c) else (np.array([],dtype=int),np.array([]))
x['distance_arret_m']=dist
x['arret_plus_proche_id']=[stops.iloc[i].arret_id if i>=0 else None for i in idx]
x['marche_estimee_min']=dist*WALK_DETOUR_FACTOR/(WALK_SPEED_KMH*1000/60)
x['marche_mesuree_min']=np.nan
walk=S('input_marche')
if len(walk):
    age=(pd.Timestamp.today().normalize()-pd.to_datetime(walk.date_observation)).dt.days
    walk=walk[age.between(0,MAX_MEASUREMENT_AGE_DAYS)]
    # Seuls les arrêts du run courant; une erreur d'identifiant bloque, sans choix silencieux.
    if not set(walk.arret_id)<=set(stops.arret_id if len(stops) else []): raise ValueError('marche: arret_id inconnu')
    if not set(walk.candidat_id)<=set(c.candidat_id): raise ValueError('marche: candidat_id inconnu')
    wm=walk.sort_values(['duree_marche_min','arret_id']).drop_duplicates('candidat_id')
    x['marche_mesuree_min']=x.candidat_id.map(wm.set_index('candidat_id').duree_marche_min)
x['transport_5min_valide']=x.marche_mesuree_min.le(WALK_MAX_MIN) & x.marche_mesuree_min.notna()
x['transport_5min_estime']=x.marche_estimee_min.le(WALK_MAX_MIN) & x.marche_estimee_min.notna()
x['statut_transport']=np.select([x.transport_5min_valide,x.marche_mesuree_min.notna(),x.transport_5min_estime],
 ['itineraire_5min_confirme','itineraire_fourni_superieur_5min','estimation_a_verifier'],default='inconnu_ou_eloigne')
# Fraîcheur et concurrence: zéro uniquement avec inventaire déclaré complet et récent.
recent=stations[stations.observation_recente==True] if len(stations) else stations
competitors=recent[recent.is_citiz==False] if len(recent) else recent
cz=recent[recent.is_citiz==True] if len(recent) else recent
x['concurrents_observes_1km']=radius_sum(c,competitors,COMPETITION_RADIUS_M)
x['citiz_observees_500m']=radius_sum(c,cz,CITIZ_SPACING_M)
_,x['distance_citiz_m']=nearest(c,cz) if len(c) else ([],[])
# Vérifier tout le bassin de concurrence, y compris les communes voisines.
from shapely.geometry import shape, Polygon
from shapely.strtree import STRtree
from shapely.ops import unary_union
from pyproj import Geod
geoms=[shape(json.loads(v)) for v in com.contour_json]; gt=STRtree(geoms)
union=unary_union(geoms); geod=Geod(ellps='WGS84')
cov=S('input_couverture_concurrence')
valid=set(cov[cov.inventaire_valide==True].commune_code) if len(cov) else set()
complete=[]
for row in c.itertuples():
    angles=np.arange(0,360,10); lons,lats,_=geod.fwd(np.full(len(angles),row.lon),np.full(len(angles),row.lat),angles,np.full(len(angles),COMPETITION_RADIUS_M))
    circle=Polygon(zip(lons,lats)); hits=gt.query(circle,predicate='intersects')
    complete.append(bool(union.covers(circle)) and all(com.iloc[int(i)].commune_code in valid for i in hits))
x['concurrence_connue']=complete
x['nb_concurrents_1km']=x.concurrents_observes_1km.where(x.concurrence_connue)
x['nb_citiz_500m']=x.citiz_observees_500m.where(x.concurrence_connue)
x=x.merge(k,on='commune_code',how='left',validate='many_to_one')
feas=S('input_faisabilite')
if len(feas): x=x.merge(feas.rename(columns={'source':'source_faisabilite'}),on='candidat_id',how='left',validate='one_to_one')
else: x['stationnement_valide']=pd.NA; x['cout_annuel_eur']=np.nan
# Sinistralité réelle Citiz: disponible séparément, pas extrapolée depuis la délinquance.
sin=S('input_sinistres_citiz')
if len(sin):
    sin['vols_pour_1000_vehicules_jours']=1000*sin.nb_vols/sin.vehicules_jours
    sin['degradations_pour_1000_vehicules_jours']=1000*sin.nb_degradations/sin.vehicules_jours
write_table(GOLD,'fact_sinistres_citiz',sin if len(sin) else pd.DataFrame(columns=['commune_code','annee','vehicules_jours','nb_vols','nb_degradations','source','vols_pour_1000_vehicules_jours','degradations_pour_1000_vehicules_jours']))


In [ ]:
if not np.isclose(sum(WEIGHTS.values()),1) or any(w<0 for w in WEIGHTS.values()): raise ValueError('Somme des poids différente de 1')
components=pd.DataFrame(index=x.index)
components['population']=(x.population_bassin/3000).clip(0,1)*100
components['croissance']=((x.croissance_annuelle+1)/3).clip(0,1)*100
components['motorisation']=(x.part_menages_sans_voiture/.5).clip(0,1)*100
# Pas de score transport fondé sur une fausse durée de marche.
components['transport']=(100*(1-x.marche_mesuree_min/10).clip(0,1)).where(x.marche_mesuree_min.notna())
# Uniquement vols automobiles. Trois indicateurs distincts: indice de contexte, pas taux total.
cols=['vol_vehicule_pour_mille','vol_dans_vehicule_pour_mille','vol_accessoire_pour_mille']
complete_crime=x[cols].notna().all(axis=1)
risk=x[cols].div(pd.Series([10,15,10],index=cols)).clip(0,1).mean(axis=1)
components['securite']=(100*(1-risk)).where(complete_crime)
components['concurrence']=(100*(1-x.nb_concurrents_1km/COMPETITORS_SATURATION)).clip(0,100)
components['congestion']=(100*(1-x.indice_congestion/CONGESTION_MAX)).clip(0,100)
# Besoin potentiel croissant jusqu'à 45 min, sans prétendre prédire des réservations.
components['mobilite']=(100*x.temps_moyen_min/45).clip(0,100)
w=pd.Series(WEIGHTS)
x['completude_score']=components.notna().mul(w).sum(axis=1)
x['score_min']=components.mul(w).sum(axis=1)  # borne si chaque inconnu valait 0
x['score_max']=x.score_min+100*(1-x.completude_score)  # borne si chaque inconnu valait 100
x['score_implantation']=x.score_min.where(x.completude_score>=.999999)
x['rang_score']=x.score_implantation.rank(ascending=False,method='min')
# Sans les fichiers manuels, 4 critères sur 8 manquent et le score strict reste vide partout.
x['score_disponible']=(x.score_min/x.completude_score).where(x.completude_score>0)
x['rang_score_disponible']=x.score_disponible.rank(ascending=False,method='min')
# Aucun "site sûr" déduit d'une moyenne communale. Une validation de terrain reste nécessaire.
known=x[['population_bassin','croissance_annuelle','nb_concurrents_1km','indice_congestion']].notna().all(axis=1)&complete_crime
pop_ok=(x.population_bassin>=MIN_POP_CATCHMENT)&((x.croissance_annuelle>=STABLE_GROWTH_PCT_AN)|(x.population_bassin>=2000))
contra=(x.indice_congestion>CONGESTION_MAX)|(x.nb_concurrents_1km>=COMPETITORS_SATURATION)|(x.vol_vehicule_pour_mille>THEFT_ALERT_PER_1000)|(x.citiz_observees_500m>0)
parking=x.stationnement_valide.fillna(False).astype(bool)
ready=known & pop_ok & x.transport_5min_valide & ~contra & parking & x.score_implantation.notna()
x['statut_candidat']=np.select([contra,ready,~known|x.marche_mesuree_min.isna(),~pop_ok],
 ['contrainte_identifiee','a_etudier_sur_le_terrain','donnees_a_completer','demande_a_confirmer'],default='validation_metier_requise')
x['motifs']=x.apply(lambda r:'; '.join(m for flag,m in [
 (pd.isna(r.population_bassin),'population locale manquante'),
 (not r.concurrence_connue,'inventaire concurrentiel incomplet ou périmé'),
 (pd.isna(r.marche_mesuree_min),'itinéraire piéton à calculer'),
 (pd.isna(r.indice_congestion),'congestion inconnue'),
 (pd.isna(r.temps_moyen_min),'durée domicile-travail inconnue'),
 (pd.isna(r.vol_vehicule_pour_mille),'vols de véhicules non diffusés ou périmés'),
 (r.citiz_observees_500m>0,'Citiz observée à moins de 500 m'),
 (pd.isna(r.stationnement_valide) or not bool(r.stationnement_valide),'stationnement à valider')]
 if flag),axis=1) if len(x) else pd.Series(dtype='string')
# Fact détaillée explicative: contributions fixes, même quand une donnée manque.
long=components.copy(); long['candidat_id']=x.candidat_id
long=long.melt(id_vars='candidat_id',var_name='critere',value_name='note')
long['poids']=long.critere.map(WEIGHTS); long['contribution']=long.note*long.poids
long['disponible']=long.note.notna()
write_table(GOLD,'dim_candidat',c)
write_table(GOLD,'gold_candidats',x)
write_table(GOLD,'gold_score_detail',long)
write_table(GOLD,'dim_arret',stops if len(stops) else pd.DataFrame(columns=['arret_id','feed_id','nom','lat','lon','commune_code']))
write_table(GOLD,'dim_station',stations if len(stations) else pd.DataFrame(columns=['station_key','operator','is_citiz','lat','lon','capacity','commune_code','date_observation','observation_recente','source']))
# Tables de paramètres et de qualité, affichées dans le rapport.
pd_params={'catchment_m':CATCHMENT_M,'competition_radius_m':COMPETITION_RADIUS_M,'walk_max_min':WALK_MAX_MIN,
 'max_auto_communes':MAX_AUTO_COMMUNES,'max_auto_candidates_per_commune':MAX_AUTO_CANDIDATES_PER_COMMUNE,
 'congestion_max':CONGESTION_MAX,'competitors_saturation':COMPETITORS_SATURATION,
 'min_population_bassin':MIN_POP_CATCHMENT,'growth_start':GROWTH_START,'growth_end':GROWTH_END,
 'study_run_id':RUN_ID,'date_calcul_utc':dt.datetime.now(dt.timezone.utc).isoformat()}
pd_params.update({'poids_'+k:v for k,v in WEIGHTS.items()})
write_table(GOLD,'gold_parametres',pd.DataFrame([{'parametre':k,'valeur':str(v)} for k,v in pd_params.items()]))
write_table(GOLD,'gold_qualite',pd.DataFrame(STATUS))
write_table(GOLD,'pipeline_run',pd.DataFrame([{'run_id':RUN_ID,'status':'complete','date_utc':dt.datetime.now(dt.timezone.utc).isoformat()}]))
print('Gold prêt. Les centres de carreaux sont des pistes, pas des places de stationnement réservables.')
